# Getting Started with LangChain for Agents

The following brings in some content from the notebooks in the LangChain intro courses, e.g., as can be found at https://github.com/langchain-ai/lca-lc-foundations

For brief review, we can use LangChain's OpenAI connector to query against the NRP API.

In [ ]:
# This assumes that you have a "keys.py" file in this directory
# with NRP_TOK assigned the value of your NRP API token (required)
import keys
NRP_TOK = keys.NRP_TOK

nrp_llm_url = "https://ellm.nrp-nautilus.io/v1"

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model = 'gpt-oss',
                   api_key = NRP_TOK,
                   base_url = nrp_llm_url,
                   # use_responses_api=False forces the classic chat.completions endpoint
                   use_responses_api=False,
                   temperature = 0.3)

After initializing our chat model, getting a response can be as easy as passing a string into the `invoke` method.

In [ ]:
response = model.invoke('What is the capital of france?')

This has wrapped the model call and given us the response object, which contains the text output to our question plus a lot more.

In [ ]:
response

To specifically get the model's answer to our query:

In [ ]:
response.content

Other pieces of the response provide useful context on the metadata level:

In [ ]:
response.response_metadata

In [ ]:
response.usage_metadata

In [ ]:
for i in response:
    print(f"{i[0]:>20} : {i[1]}\n")

# Another simpler approach

In [ ]:
from langchain.chat_models import init_chat_model

In [ ]:
model = init_chat_model(model="gpt-oss",
                        base_url=nrp_llm_url,
                        api_key=NRP_TOK)

In [ ]:
model.invoke("What is the capital of CA?")

# Creating an agent

In [ ]:
from langchain.agents import create_agent

If you have an OPENAIKEY in your environment, you can do something like:
* `agent = create_agent('gpt-5')`

But here we need to use the chat model from above.

In [ ]:
agent = create_agent(model=model)

There are a couple things we need to modify relative to invoking the model.

In [ ]:
# This will give an ERROR
agent.invoke("What is the capital of CA?")

If you want to check, this will also not work:

In [ ]:
# This will give an ERROR!
agent = create_agent(model="gpt-oss",
                     base_url=nrp_llm_url,
                     api_key=NRP_TOK)

We need to be a little careful in using a model with a different endpoint.  Here is a way to see some help documentation:

In [ ]:
help(create_agent)

A direct chat model instance (such as our ChatOpenAI instance above) is acceptable for the agent model.

In [ ]:
model = ChatOpenAI(model = 'gpt-oss',
                 api_key = NRP_TOK,
                 base_url = nrp_llm_url,
                 # use_responses_api=False forces the classic chat.completions endpoint
                 use_responses_api=False,
                 temperature = 0.3)

In [ ]:
agent = create_agent(model=model)

As far as `agent.invoke`, let's look at help while we're at it:

In [ ]:
help(agent.invoke)

Notice that there's a lot of mention of "graphs".  This is because we're working with LangGraph too.  

The `create_agent` function from `langchain.agents` is built on top of LangGraph, which provides the underlying graph-based execution engine for agents.

Our book has an example with the following:
* `from langchain.agents import AgentExecutor, create_react_agent`

I mentioned previously this needed to be changed to:
* `from langchain_classic.agents import AgentExecutor, create_react_agent`

We will stick to `langchain.agents.create_agent` but revisit the book example.

LangChain has been undergoing lots of modifications since the book came out.
* "LangChain and LangGraph Agent Frameworks Reach v1.0 Milestones": https://blog.langchain.com/langchain-langgraph-1dot0/
* LangGraph vs LangChain vs DeepAgents: https://docs.langchain.com/oss/python/concepts/products

If you turn to other tutorials, you may also see a variety of other frameworks pop up, including LlamaIndex and smolagents.  (Also LangSmith and LangServe -- these are frameworks useful for productionizing work)

| Dimension                               | LangChain                                                                                         | LangGraph                                                                                                 | LlamaIndex                                                                                       | smolagents                                                                                                           |
| --------------------------------------- | ------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------------- |
| **Quick mental model**                  | General LLM app & agent framework. Gives you lots of building blocks (chains, tools, agents).   | Workflow / state machine for agents. You draw the graph of steps and LangGraph runs it.                 | Data / RAG framework. Focused on connecting LLMs to your own data (docs, DBs, etc.).           | "Tiny agent library." Lets models write & run Python code to do things, with minimal abstractions.                   |
| **Main focus**                          | Build LLM‑powered apps and agents with many integrations (models, vector DBs, tools).             | Build stateful, controllable, often multi‑step or multi‑agent workflows over time.                      | Make it easy to ingest, index, retrieve, and query your private data using LLMs.                 | Minimal code‑centric agents that call tools and execute code, often via sandboxes.                                   |
| **Good first use case**       | Simple tool‑using chatbot (e.g., question answering + calling a calculator or search API).        | A multi‑step pipeline: plan -> retrieve data -> call tool -> summarize, with clear steps in a graph.         | "Chat with your PDFs / SQL DB" RAG assistant over a dataset.                              | A research or coding helper that writes small Python snippets to answer questions or analyze data.                   |
| **Agent style**                         | Provides built‑in agent types (e.g., ReAct‑style) that decide when to call tools, how often, etc. | You explicitly define nodes (functions / LLM calls) and edges; the system handles control flow and state. | Data tools as first‑class citizens: query engines, indexes, and agents that use them as tools. | Code‑first agents: the model often responds with code that gets executed, plus simple tool‑calling variants.         |
| **Data / RAG support**                  | Good RAG support (loaders, retrievers, vector stores), but it’s one part of a broader framework.  | Not a data library by itself; usually wraps RAG steps you build with LangChain or custom tools.           | RAG‑first: indexing, retrieval, query orchestration, and evaluation are core to the library.     | No special RAG layer; you wrap search / DB / RAG systems as tools the agent can use.                                 |

# Back to work

Our previous work with roles (e.g. system, user) as elements of model messages take on a new flavor.

In the above you might have noticed `AIMessage`, and we can also work with `SystemMessage` and `HumanMessage`

In [ ]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

Now we get to a working `agent.invoke`

In [ ]:
response = agent.invoke(
    {'messages': [HumanMessage(content='What is the capital of france?')]}
)

In [ ]:
response

In [ ]:
response['messages'][0]

In [ ]:
response['messages'][1]

In [ ]:
# Getting the final response output
response['messages'][-1].content

Just as before, we can use system messages to provide higher level instructions/guidance to the LLM.

In [ ]:
response = agent.invoke(
    {'messages': [SystemMessage(content='write one-word responses'),
                  HumanMessage(content='What is the capital of france?')]}
)

In [ ]:
response

In [ ]:
# Getting the final response output
response['messages'][-1].content

By the way, this does also work for `model.invoke` (as opposed to `agent.invoke`) but **not** in the same way

In [ ]:
# will give an error
response = model.invoke(
    {'messages': [SystemMessage(content='write one-word responses'),
                  HumanMessage(content='What is the capital of france?')]}
)

In [ ]:
# will NOT give an error
response = model.invoke(
    [SystemMessage(content='write one-word responses'),
     HumanMessage(content='What is the capital of france?')]
)

In [ ]:
response

Turning back to `agent`, there are some useful things to get started on for conversations and agent use.

We can chain messages to create something like a conversation.

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of CA?"),
                  AIMessage(content="The capital of CA is Los Angeles."),
                  HumanMessage(content="Interesting, how did Los Angeles become the capital?")]}
)

In [ ]:
response['messages'][-1].content

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
                  AIMessage(content="The capital of the Moon is Luna City."),
                  HumanMessage(content="Interesting, tell me more about Luna City")]}
)

In [ ]:
from IPython.display import display, Markdown

In [ ]:
text = response['messages'][-1].content
display(Markdown(text))

Too much info??

## Streaming example

`agent.stream` will allow us to stream the output instead of getting a massive string all at once.

Specifically, it will return two outputs in chunked intervals, with the first being a message chunk that contains response content.

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token
    
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

# Having conversations with memory

Another useful conversational element is retaining memory of content.  ConversationBufferMemory and etc have been deprecated, but a currently useful way to do this is with LangGraph's InMemoryStore.



In [ ]:
question = HumanMessage(content="My name is Ben and my favourite color is aqua.")

response = agent.invoke(
    {"messages": [question]} 
)

response['messages'][-1].content

In [ ]:
question = HumanMessage(content="What is my favourite color?")

response = agent.invoke(
    {"messages": [question]} 
)

response['messages'][-1].content

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  

We need to initialize the model with checkpointer.

In [ ]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  
)

We also need to specify a thread ID number so that the LLM can keep track of a continous discussion thread.

In [ ]:
question = HumanMessage(content="My name is Ben and my favourite color is aqua.")

response = agent.invoke(
    {"messages": [question]},
    {"configurable": {"thread_id": "1"}}
)

response['messages'][-1].content

In [ ]:
question = HumanMessage(content="What is my favourite color?")

response = agent.invoke(
    {"messages": [question]},
    {"configurable": {"thread_id": "1"}}
)

response['messages'][-1].content

For production use cases, it's better to use a persistent store such as a database.